In [ ]:
dbutils.widgets.text("pipeline_name", "ops_pipeline")
dbutils.widgets.text("connection_name", "ops")
dbutils.widgets.text("ingestion_dir", "<ingestion_dir>")
dbutils.widgets.text("dlt_ingest_limit", "-1")
dbutils.widgets.text("dev_mode", "false")
dbutils.widgets.text("dlt_env", "{}")

In [ ]:
INGESTION_DIR = dbutils.widgets.get("ingestion_dir")

In [ ]:
%pip install -q -c $INGESTION_DIR/../.github/managed-constraints.txt "dlt[databricks]" pyarrow vibedata-dlt-core vibedata-dlt-databricks-notebook -r $INGESTION_DIR/requirements.txt

In [ ]:
dbutils.library.restartPython()

In [ ]:
from vibedata.dlt.databricks import DatabricksUcSecretsProvider, databricks_files_api, setup_environment

import dlt

setup_environment()
# Resolves nothing unless config.toml names a Unity Catalog secrets store.
dlt.config.register_provider(DatabricksUcSecretsProvider())

In [ ]:
import json
import os

from vibedata.dlt.core import dataset, persist_audit

from sources.ops import ops_source
from sources.tlc import yellow_trips

PIPELINE_NAME = dbutils.widgets.get("pipeline_name")
CONNECTION_NAME = dbutils.widgets.get("connection_name")
LIMIT = int(dbutils.widgets.get("dlt_ingest_limit"))

os.environ.update(json.loads(dbutils.widgets.get("dlt_env")))

destination = databricks_files_api()

pipeline = dlt.pipeline(
    pipeline_name=PIPELINE_NAME,
    destination=destination,
    dataset_name=dataset("ops_raw"),
    dev_mode=dbutils.widgets.get("dev_mode") == "true",
)

# The operator's records are derived from the yellow trips of the window the tlc pipeline
# loads. They are read in memory here and never written: this loads the eight ops_raw tables.
# Each run regenerates the whole operator dataset, so every resource is a full replace.
source = ops_source.clone(name=CONNECTION_NAME, section=CONNECTION_NAME)(trips=yellow_trips)
source.add_limit(LIMIT, count_rows=True)

# `tables` stays evolve: every Intent's prefixed schemas start empty (design record D-13, which defers to tlc's D-11).
# apply_hints is per-resource; a source-level call fails at run time.
for resource in source.selected_resources.values():
    resource.apply_hints(
        write_disposition="replace",
        schema_contract={"columns": "freeze", "tables": "evolve", "data_type": "freeze"},
    )

try:
    load_info = pipeline.run(source)
except Exception as exc:
    persist_audit(pipeline, pipeline.last_trace, error_message=str(exc))
    raise
print(load_info)
print("audit:", persist_audit(pipeline, pipeline.last_trace))

unfinished = [package for package in load_info.load_packages if package.state != "loaded"]
if load_info.has_failed_jobs or unfinished:
    raise RuntimeError("dlt load failed")

try:
    with pipeline.sql_client() as client:
        loaded_catalog = client.execute_sql("select current_catalog()")[0][0]
except Exception as exc:
    print("destination-identity unavailable:", exc)
else:
    dbutils.notebook.exit("destination-identity: " + json.dumps({"type": load_info.destination_type, "database": loaded_catalog}))